# Mechanistic Investigation

**Goal:** Understand the *mechanism* behind the modality gap — not just *what* happens, but *why*.

## Three Probes

### 1. Attention Analysis
- Where do text tokens "look" at image tokens?
- Do some attention heads specialize in cross-modal attention?
- Does high image-attention correlate with errors?

### 2. OCR Quality Isolation
- Ask the model to just *transcribe* the image (no math)
- Compare transcription accuracy: are numbers read correctly?
- This separates OCR errors from reasoning errors definitively

### 3. Representation Similarity
- How similar are the internal representations for text-only vs image input?
- If highly similar → problem is downstream (reasoning)
- If very different → problem is upstream (encoding)

**Runtime:** ~30-60 min on subset of 50 problems

In [ ]:
!pip install torch transformers datasets tqdm pillow bitsandbytes matplotlib seaborn --quiet

In [ ]:
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /content/repo 2>/dev/null || echo 'exists'
import sys
sys.path.insert(0, '/content/repo')

from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/vlm_research_results/mechanistic'
!mkdir -p {OUTPUT_DIR}

In [ ]:
import os, torch, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm
from PIL import Image

from src.models import VLMModel
from src.rendering import render_all_images, load_image
from src.evaluation import answers_match
from src.mechanistic import (
    extract_attention_maps, plot_attention_to_image,
    plot_cross_modal_attention_heatmap,
    compare_ocr_output, compare_hidden_states,
    batch_hidden_state_analysis,
)

torch.manual_seed(42)
N = 50  # subset for mechanistic analysis

ds = load_dataset('gsm8k', 'main', split='test').select(range(N))
questions = list(ds['question'])
references = list(ds['answer'])

IMAGE_DIR = os.path.join(OUTPUT_DIR, 'images')
render_all_images(questions, IMAGE_DIR)
print(f'Ready: {N} problems')

In [ ]:
# Load model (use smaller model for interpretability — full weights fit in memory)
MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'
vlm = VLMModel(MODEL_NAME, 'qwen', max_new_tokens=256, torch_dtype='bfloat16')
vlm.load()
short = MODEL_NAME.split('/')[-1]

## 1. Attention Analysis

In [ ]:
# Extract attention maps for a sample problem
sample_img = load_image(0, IMAGE_DIR)
sample_prompt = 'Read the math problem and solve it step by step. End with #### <answer>.'

try:
    attn_data = extract_attention_maps(
        vlm.model, vlm.processor, sample_img, sample_prompt, 'qwen')
    print(f'Layers: {attn_data["n_layers"]}, Heads: {attn_data["n_heads"]}, '
          f'Seq len: {attn_data["seq_len"]}')
    print(f'Image tokens: {attn_data["image_token_range"]}')
except Exception as e:
    print(f'Attention extraction not supported for this model: {e}')
    attn_data = None

In [ ]:
if attn_data:
    # Plot attention to image tokens at different layers
    for layer_idx in [-1, -4, 0]:  # last, middle-ish, first
        plot_attention_to_image(
            attn_data, layer=layer_idx,
            save_path=os.path.join(OUTPUT_DIR, f'{short}_attn_layer{layer_idx}.png'))
    
    # Cross-modal attention by head
    plot_cross_modal_attention_heatmap(
        attn_data, layer=-1,
        save_path=os.path.join(OUTPUT_DIR, f'{short}_cross_modal_heads.png'))

## 2. OCR Quality Isolation

In [ ]:
# Test OCR accuracy on subset
ocr_results = []
for i in tqdm(range(min(N, 30)), desc='OCR test'):
    img = load_image(i, IMAGE_DIR)
    try:
        result = compare_ocr_output(vlm.model, vlm.processor, questions[i], img, 'qwen')
        result['problem_id'] = i
        ocr_results.append(result)
    except Exception as e:
        ocr_results.append({'problem_id': i, 'error': str(e)})

ocr_df = pd.DataFrame(ocr_results)
valid_ocr = ocr_df[ocr_df['numbers_correct'].notna()]

if len(valid_ocr) > 0:
    pct_correct = valid_ocr['numbers_correct'].mean() * 100
    avg_overlap = valid_ocr['word_overlap'].mean() * 100
    print(f'\nOCR Results ({len(valid_ocr)} problems):')
    print(f'  Numbers correctly read: {pct_correct:.1f}%')
    print(f'  Word overlap: {avg_overlap:.1f}%')
    
    # Show problems where numbers were misread
    misread = valid_ocr[valid_ocr['numbers_correct'] == False]
    if len(misread) > 0:
        print(f'\n  Problems with misread numbers: {len(misread)}')
        for _, row in misread.head(5).iterrows():
            print(f'    Problem {row["problem_id"]}:')
            print(f'      Missing: {row["numbers_missing"]}')
            print(f'      Hallucinated: {row["numbers_hallucinated"]}')
    else:
        print('\n  All numbers read correctly! OCR is NOT the bottleneck.')

## 3. Representation Similarity

In [ ]:
# Compare hidden states between text and image conditions
images = [load_image(i, IMAGE_DIR) for i in range(min(N, 30))]

hs_df = batch_hidden_state_analysis(
    vlm.model, vlm.processor, questions[:30], images, 'qwen', max_problems=30)

valid_hs = hs_df[hs_df['cosine_similarity'].notna()]
if len(valid_hs) > 0:
    print(f'\nRepresentation Similarity ({len(valid_hs)} problems):')
    print(f'  Mean cosine similarity: {valid_hs["cosine_similarity"].mean():.4f}')
    print(f'  Std:  {valid_hs["cosine_similarity"].std():.4f}')
    print(f'  Min:  {valid_hs["cosine_similarity"].min():.4f}')
    print(f'  Max:  {valid_hs["cosine_similarity"].max():.4f}')
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Histogram of cosine similarities
    axes[0].hist(valid_hs['cosine_similarity'], bins=20, color='#4C72B0', edgecolor='black')
    axes[0].set_xlabel('Cosine Similarity')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Text vs Image Representation Similarity')
    axes[0].axvline(x=valid_hs['cosine_similarity'].mean(), color='red', linestyle='--',
                    label=f'Mean: {valid_hs["cosine_similarity"].mean():.3f}')
    axes[0].legend()
    
    # L2 distance distribution
    axes[1].hist(valid_hs['l2_distance'], bins=20, color='#C44E52', edgecolor='black')
    axes[1].set_xlabel('L2 Distance')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Representation Divergence')
    
    plt.suptitle(f'Internal Representation Analysis — {short}', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{short}_representation_similarity.png'), dpi=150)
    plt.show()
    
    mean_sim = valid_hs['cosine_similarity'].mean()
    if mean_sim > 0.9:
        print('\n→ High similarity: representations are nearly identical.')
        print('  The modality gap is likely in DECODING/REASONING, not ENCODING.')
    elif mean_sim > 0.7:
        print('\n→ Moderate similarity: some encoding differences exist.')
        print('  Both encoding and reasoning contribute to the gap.')
    else:
        print('\n→ Low similarity: modalities produce very different representations.')
        print('  The gap starts at ENCODING — vision features differ fundamentally.')

In [ ]:
vlm.unload()

# Save all results
if len(valid_ocr) > 0:
    ocr_df.to_csv(os.path.join(OUTPUT_DIR, f'{short}_ocr_results.csv'), index=False)
if len(valid_hs) > 0:
    hs_df.to_csv(os.path.join(OUTPUT_DIR, f'{short}_hidden_states.csv'), index=False)

print(f'\nAll mechanistic results saved to: {OUTPUT_DIR}')